# Sponsored Search Ranking — Colab Training
**Two-stage sponsored search:** TF-Ranking listwise model + FAISS HNSW retrieval

Runtime: GPU (T4) recommended. CPU works but ~3× slower.

Steps:
1. Clone repo + install dependencies
2. Generate synthetic dataset
3. Train TF-Ranking model (LambdaLoss)
4. Evaluate NDCG@10, MRR
5. Build FAISS index
6. Download model artifacts

In [17]:
import os, sys, subprocess
os.chdir('/content/sponsored-search-ranking/sponsored-search-ranking/sponsored-search-ranking')
sys.path.insert(0, os.getcwd())
os.makedirs('models', exist_ok=True)

# ── Data ──────────────────────────────────────────────────────────────────────
exec(open('data/synthetic_data.py').read())
df = generate_dataset(n_queries=10000, ads_per_query=10)
train_df, val_df, test_df = train_val_test_split(df)
print(f'Dataset: {len(df):,} rows')

# ── Imports ───────────────────────────────────────────────────────────────────
import tensorflow as tf
import numpy as np
print('TF:', tf.__version__)

# ── Model (no tensorflow_ranking dependency) ──────────────────────────────────
FEATURE_NAMES = ['bm25_score','semantic_sim','historical_ctr','bid_cpm','query_freq_log','position_bias']
NUM_FEATURES = len(FEATURE_NAMES)
LIST_SIZE = 10

class SponsoredSearchRanker(tf.keras.Model):
    def __init__(self, hidden_units=(256,128,64), dropout_rate=0.2, **kwargs):
        super().__init__(**kwargs)
        layers = []
        for units in hidden_units:
            layers += [tf.keras.layers.Dense(units),
                       tf.keras.layers.BatchNormalization(),
                       tf.keras.layers.ReLU(),
                       tf.keras.layers.Dropout(dropout_rate)]
        layers.append(tf.keras.layers.Dense(1))
        self.tower = tf.keras.Sequential(layers)
        self.ndcg_10 = tf.keras.metrics.Mean(name='ndcg@10')
        self.ndcg_5  = tf.keras.metrics.Mean(name='ndcg@5')
        self.mrr_m   = tf.keras.metrics.Mean(name='mrr')

    def call(self, inputs, training=False):
        b = tf.shape(inputs)[0]
        flat = tf.reshape(inputs, [-1, inputs.shape[-1]])
        scores = self.tower(flat, training=training)
        return tf.reshape(scores, [b, -1])

    def _ndcg(self, labels, scores, k=10):
        top_k = tf.minimum(k, tf.shape(labels)[1])
        _, idx = tf.math.top_k(scores, k=top_k)
        gathered = tf.gather(labels, idx, batch_dims=1)
        gains = tf.pow(2.0, gathered) - 1.0
        discounts = tf.math.log(tf.cast(tf.range(1, top_k+1), tf.float32)+1.0)/tf.math.log(2.0)
        dcg = tf.reduce_sum(gains/discounts, axis=1)
        ideal = tf.sort(labels, direction='DESCENDING')[:, :top_k]
        idcg = tf.reduce_sum((tf.pow(2.0, ideal)-1.0)/discounts, axis=1)
        return tf.reduce_mean(dcg/(idcg+1e-10))

    def _mrr(self, labels, scores):
        _, idx = tf.math.top_k(scores, k=tf.shape(labels)[1])
        gathered = tf.gather(labels, idx, batch_dims=1)
        hits = tf.cast(gathered > 0.5, tf.float32)
        positions = tf.cast(tf.range(1, tf.shape(labels)[1]+1), tf.float32)
        rr = hits / positions
        first_hit = tf.reduce_max(rr * tf.cast(
            tf.cumsum(tf.cast(hits>0, tf.int32), axis=1) <= 1, tf.float32), axis=1)
        return tf.reduce_mean(first_hit)

    def train_step(self, data):
        features, labels, weights = data
        with tf.GradientTape() as tape:
            scores = self(features, training=True)
            loss = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(
                labels=tf.nn.softmax(labels * weights), logits=scores))
        grads = tape.gradient(loss, self.trainable_variables)
        grads, _ = tf.clip_by_global_norm(grads, 1.0)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        self.ndcg_10.update_state(self._ndcg(labels, scores, 10))
        self.ndcg_5.update_state(self._ndcg(labels, scores, 5))
        self.mrr_m.update_state(self._mrr(labels, scores))
        return {'loss': loss, 'ndcg@10': self.ndcg_10.result(),
                'ndcg@5': self.ndcg_5.result(), 'mrr': self.mrr_m.result()}

    def test_step(self, data):
        features, labels, weights = data
        scores = self(features, training=False)
        loss = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(
            labels=tf.nn.softmax(labels * weights), logits=scores))
        self.ndcg_10.update_state(self._ndcg(labels, scores, 10))
        self.ndcg_5.update_state(self._ndcg(labels, scores, 5))
        self.mrr_m.update_state(self._mrr(labels, scores))
        return {'loss': loss, 'ndcg@10': self.ndcg_10.result(),
                'ndcg@5': self.ndcg_5.result(), 'mrr': self.mrr_m.result()}

    @property
    def metrics(self):
        return [self.ndcg_10, self.ndcg_5, self.mrr_m]

# ── Dataset builder ───────────────────────────────────────────────────────────
def build_dataset(df, list_size=10, batch_size=512, shuffle=True):
    query_ids = df['query_id'].unique()
    features_list, labels_list, weights_list = [], [], []
    for qid in query_ids:
        qdf = df[df['query_id']==qid].head(list_size)
        pad = list_size - len(qdf)
        feat = qdf[['bm25_score','semantic_sim','historical_ctr',
                     'bid_cpm','query_freq','position_bias']].values.astype(np.float32)
        feat[:,4] = np.log1p(feat[:,4])  # log-normalize query_freq
        lbl = qdf['label'].values.astype(np.float32)
        wt  = qdf['position_bias'].values.astype(np.float32)
        if pad > 0:
            feat = np.vstack([feat, np.zeros((pad, 6), dtype=np.float32)])
            lbl  = np.concatenate([lbl,  np.zeros(pad, dtype=np.float32)])
            wt   = np.concatenate([wt,   np.zeros(pad, dtype=np.float32)])
        features_list.append(feat)
        labels_list.append(lbl)
        weights_list.append(wt)
    fa = np.stack(features_list)
    la = np.stack(labels_list)
    wa = np.stack(weights_list)
    ds = tf.data.Dataset.from_tensor_slices((fa, la, wa))
    if shuffle:
        ds = ds.shuffle(len(query_ids))
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# ── Build datasets ────────────────────────────────────────────────────────────
train_ds = build_dataset(train_df, batch_size=512, shuffle=True)
val_ds   = build_dataset(val_df,   batch_size=512, shuffle=False)
test_ds  = build_dataset(test_df,  batch_size=512, shuffle=False)
print(f'Train batches: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

# ── Build model ───────────────────────────────────────────────────────────────
model = SponsoredSearchRanker(hidden_units=(256,128,64), dropout_rate=0.2)
model.compile(optimizer=tf.keras.optimizers.Adam(
    tf.keras.optimizers.schedules.CosineDecayRestarts(1e-3, len(train_ds)*5)))
model(tf.zeros((1, 10, 6)))
print(f'Parameters: {model.count_params():,}')

# ── Train ─────────────────────────────────────────────────────────────────────
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        'models/ranker_best.weights.h5', monitor='val_ndcg@10',
        mode='max', save_best_only=True, save_weights_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_ndcg@10', patience=5, mode='max',
        restore_best_weights=True, verbose=1),
]
history = model.fit(train_ds, validation_data=val_ds,
                    epochs=20, callbacks=callbacks, verbose=1)

# ── Evaluate ──────────────────────────────────────────────────────────────────
results = model.evaluate(test_ds, verbose=1)
print('\n=== TEST RESULTS ===')
for name, val in zip(['loss','ndcg@10','ndcg@5','mrr'], results):
    print(f'  {name}: {val:.4f}')
print(f'\nBest val ndcg@10: {max(history.history.get("val_ndcg@10",[0])):.4f}')

Generated 50,000 rows (5000 queries × 10 ads)
Click rate: 0.219
Columns: ['query_id', 'query_text', 'ad_id', 'ad_text', 'label', 'bm25_score', 'semantic_sim', 'historical_ctr', 'bid_cpm', 'position_bias', 'query_freq']
Train: 35,000 | Val: 7,500 | Test: 7,500
Saved to data/synthetic_search_data.csv
Generated 100,000 rows (10000 queries × 10 ads)
Click rate: 0.219
Columns: ['query_id', 'query_text', 'ad_id', 'ad_text', 'label', 'bm25_score', 'semantic_sim', 'historical_ctr', 'bid_cpm', 'position_bias', 'query_freq']
Train: 70,000 | Val: 15,000 | Test: 15,000
Dataset: 100,000 rows
TF: 2.19.0
Train batches: 14 | Val: 3 | Test: 3
Parameters: 44,801
Epoch 1/20
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 418ms/step - loss: 2.4036 - mrr: 0.4923 - ndcg@10: 0.5750 - ndcg@5: 0.4166
Epoch 1: val_ndcg@10 improved from None to 0.62152, saving model to models/ranker_best.weights.h5

Epoch 1: finished saving model to models/ranker_best.weights.h5
14/14 ━━━━━━━━━━━━━━━━━━━━ 15s 569ms/step - loss: 2.3567 - mrr: 0.51